# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane 3: Structured Content Archetype Clustering.

I read notebook 01's EDA that already showed CTR falls off sharply by position, and notebook 02 that showed a simple hand rule can match a learned model on ranking risk. What neither showed is whether pages reduce to one number at all. Below, ctr and engagement_rate correlate at only ~0.09 across visible pages. Two behavioral metrics that barely move together. That means a page can be doing well on one axis and badly on another at the same time, which a single score (or a single ranked queue) would blur together. Clustering is built for exactly this: find the small number of shapes a page's behavior actually takes (e.g. good position but no engagement, weak position but strong engagement, stale but still visible) so an editor can react to the type of page instead of reading every metric on every page one at a time.

In [ ]:
# This cell is for CODE (numbers, a query, a check).

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()

print("Pages usable for archetype work:", len(visible), "/", len(df))
print("corr(ctr, engagement_rate):", round(visible["ctr"].corr(visible["engagement_rate"]), 3))
print("corr(ctr, scroll_rate):", round(visible["ctr"].corr(visible["scroll_rate"]), 3))



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision it improves: which treatment—protect, improve CTR, rewrite, merge, prune, or monitor—a batch of pages that share a behavioral profile should get, instead of a content strategist deciding page-by-page from raw metrics.

Who acts, and how: a content strategist / reviewing editor with limited weekly review capacity. Today they scan a spreadsheet of metrics per page; the output here is a short list of named archetypes plus which pages belong to each, so they can triage a cluster at a time.

Cost of a wrong call:

Labeling a real "rising star" (early, still building) as prune-worthy loses traffic that was about to recover on its own. It is reversible, but wasteful.
Labeling a "hidden gem" (strong engagement, weak position) as "protect / leave alone" misses an easy win because it needed a ranking push, not to be ignored.
Spending review time re-checking already-strong "champion" pages that needed no action at all, so the direct cost is editor hours, not a broken page.
None of these costs are irreversible or safety-critical. This is decision-support that saves review time and surfaces easy wins, not an automated action that touches a live page.

Why data/ML helps: a page's situation is described by several loosely-related signals at once (position, CTR, engagement, freshness, volume) that don't reduce to one obvious if/else. The near-zero correlation above is the evidence. Writing that by hand as nested rules for every combination would be brittle and would miss groupings nobody thought to write a rule for. Clustering finds the groupings from the data itself; a human still names and validates them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).

# Section 2 is reasoning, not a new number because it leans on the correlation check from Section 1.
# No new query needed here.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Three numbers from the starter CSV, each pointing at real structure a single score would hide:

22,006 / 30,000 pages have real position data and enough traffic to trust (`impressions_90d
= 100, avg_position > 0`). It is a large enough usable base for clustering.

ctr and engagement_rate correlate at only 0.093 (Section 1). These pages don't move together on these two axes, so a page can rank fine but disengage users, or the reverse.
engagement_rate == 0 for 63.8% of visible pages, but that rate ranges from 62.9% (keyword articles) to 98.6% (comparison articles). This is too content-type-dependent to be "true zero engagement" across the board; it looks like a measurement-coverage gap, not a behavior. I'm flagging it now because it's exactly the kind of thing the flyrank-data skill warns about: a naive fillna(0) or trusting these zeros at face value would quietly inject a content-type signal into any cluster built on this column. Any archetype work needs a has_engagement_data flag before engagement_rate is used as a real feature.

In [ ]:
# This cell is for CODE (numbers, a query, a check).

zero_engagement = (visible["engagement_rate"] == 0).sum()
print(f"engagement_rate == 0: {zero_engagement} / {len(visible)} "
      f"({zero_engagement/len(visible)*100:.1f}%)")
print()
print("Share of rows with engagement_rate == 0, by content_type:")
print(visible.groupby("content_type")["engagement_rate"].apply(lambda s: (s == 0).mean()).round(3))


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can claim**:

These will be observed behavioral archetypes over a 90-day trailing window in this anonymized starter sample (later, the warehouse release),a descriptive lens for triage, not a proof of cause. Cluster membership is directional ("this page behaves like a hidden gem"), not certain, and every cluster gets human-inspected before it's named or acted on.

**Can't claim**:

- **Not semantic clustering**. The data has no article text, only metrics, buckets, and token/word counts. Calling metric clustering "semantic" would claim something I didn't do (explicit lane guide warning).
- **Not a prediction of future decline or recovery**. Clustering here is a snapshot lens, not a time-aware model. That's a different lane/task.
- **Not proof that any archetype's editorial fix will work, and never a claim about Google's algorithm or ranking factors**. This data is observational only.
Not treating engagement_rate == 0 as literal zero engagement until the missingness gap above is handled with a coverage flag, per Section 3.

In [ ]:
# This cell is for CODE (numbers, a query, a check).

# Section 4 is a scoping statement -- no new query needed here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.